In [1]:
print("Hi aii")

Hi aii


In [2]:
import os
import json
import math
import re
import sqlite3
import time
from collections import Counter, defaultdict
from typing import Any, TypedDict

import numpy as np
from openai import OpenAI

In [3]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY before running this notebook: export OPENAI_API_KEY=sk-...")

client = OpenAI(api_key=OPENAI_API_KEY)
EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-4o-mini")

print(f"OpenAI ready. Chat model: {CHAT_MODEL}; embedding model: {EMBED_MODEL}")

OpenAI ready. Chat model: gpt-4o-mini; embedding model: text-embedding-3-small


In [4]:
CORPUS = [
    {
        "id": "pods",
        "source": "pods.md",
        "text": "A Kubernetes Pod is the smallest deployable unit. Containers in a Pod share network, storage volumes, and lifecycle. Use kubectl describe pod and kubectl logs to debug Pod behavior.",
    },
    {
        "id": "deployment",
        "source": "deployment.md",
        "text": "A Deployment manages ReplicaSets and rolling updates. Use kubectl rollout status deployment/nginx to watch progress and kubectl rollout undo deployment/nginx to roll back a bad release.",
    },
    {
        "id": "service",
        "source": "service.md",
        "text": "A Service gives stable networking for Pods. ClusterIP is internal, NodePort exposes a port on nodes, and LoadBalancer asks the cloud provider for an external endpoint.",
    },
    {
        "id": "probes",
        "source": "probes.md",
        "text": "Readiness probes decide when a Pod can receive traffic. Liveness probes restart stuck containers. Startup probes protect slow-starting applications from premature restarts.",
    },
    {
        "id": "secrets",
        "source": "secrets.md",
        "text": "Kubernetes Secrets store sensitive values such as passwords, tokens, and keys. Enable encryption at rest, restrict RBAC, and avoid printing secret data in logs.",
    },
    {
        "id": "taints",
        "source": "taints.md",
        "text": "Taints repel Pods from nodes. Tolerations allow selected Pods to schedule onto tainted nodes. A NoSchedule taint blocks Pods that lack a matching toleration.",
    },
    {
        "id": "hpa",
        "source": "hpa.md",
        "text": "The HorizontalPodAutoscaler increases or decreases replicas based on CPU, memory, or custom metrics so an application can handle more traffic automatically.",
    },
]

NOISE_DOCS = [
    {"id": "noise-cache", "source": "cache-paper.md", "text": "Cache-aware matrix multiplication improves locality in CPU memory hierarchies and reduces cache misses."},
    {"id": "noise-graph", "source": "graph-paper.md", "text": "Graph partitioning algorithms optimize edge cuts in distributed computation workloads."},
]

print(f"Inline corpus loaded: {len(CORPUS)} K8s snippets + {len(NOISE_DOCS)} noise snippets")

Inline corpus loaded: 7 K8s snippets + 2 noise snippets


In [5]:
def embed_texts(texts: list[str]) -> list[list[float]]:
    response = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [item.embedding for item in response.data]

In [6]:

def cosine(a: list[float], b: list[float]) -> float:
    av = np.array(a, dtype=float)
    bv = np.array(b, dtype=float)
    denom = np.linalg.norm(av) * np.linalg.norm(bv)
    return float(np.dot(av, bv) / denom) if denom else 0.0

In [7]:
def dense_search(question: str, docs: list[dict[str, str]], top_k: int = 3) -> list[dict[str, Any]]:
    vectors = embed_texts([question] + [d["text"] for d in docs])
    qv, doc_vectors = vectors[0], vectors[1:]
    ranked = []
    for doc, dv in zip(docs, doc_vectors):
        ranked.append({**doc, "score": cosine(qv, dv)})
    return sorted(ranked, key=lambda d: d["score"], reverse=True)[:top_k]



In [8]:
def answer_with_context(question: str, chunks: list[dict[str, Any]]) -> str:
    context = "\n\n".join(f"SOURCE: {c['source']}\n{c['text']}" for c in chunks)
    messages = [
        {"role": "system", "content": "Answer only from the provided Kubernetes context. Cite source names inline."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    response = client.chat.completions.create(model=CHAT_MODEL, messages=messages, temperature=0)
    return response.choices[0].message.content

In [9]:
def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-zA-Z0-9_:-]+", text.lower())

In [10]:
def sparse_score(question: str, doc: dict[str, str]) -> float:
    q_terms = Counter(tokenize(question))
    d_terms = Counter(tokenize(doc["text"]))
    return sum(q_terms[t] * d_terms[t] for t in q_terms)

In [11]:
def sparse_search(question: str, docs: list[dict[str, str]], top_k: int = 3) -> list[dict[str, Any]]:
    ranked = [{**doc, "score": sparse_score(question, doc)} for doc in docs]
    return sorted(ranked, key=lambda d: d["score"], reverse=True)[:top_k]

In [12]:
def rrf_fuse(result_lists: list[list[dict[str, Any]]], k: int = 60, top_k: int = 3) -> list[dict[str, Any]]:
    scores: dict[str, float] = defaultdict(float)
    docs: dict[str, dict[str, Any]] = {}
    for results in result_lists:
        for rank, doc in enumerate(results, start=1):
            scores[doc["id"]] += 1 / (k + rank)
            docs[doc["id"]] = doc
    fused = [{**docs[doc_id], "rrf_score": score} for doc_id, score in scores.items()]
    return sorted(fused, key=lambda d: d["rrf_score"], reverse=True)[:top_k]

In [15]:
exact_query = "kubectl rollout undo deployment/nginx"
print("Sparse results")
for c in sparse_search(exact_query, CORPUS + NOISE_DOCS):
    print(c["source"], c["score"])

print("\nDense results")
for c in dense_search(exact_query, CORPUS + NOISE_DOCS):
    print(c["source"], f"{c['score']:.3f}")

Sparse results
deployment.md 10
pods.md 2
service.md 0

Dense results
deployment.md 0.681
pods.md 0.301
secrets.md 0.243


In [16]:
hybrid_query = "How do I send traffic to Pods from inside the cluster?"
dense = dense_search(hybrid_query, CORPUS + NOISE_DOCS, top_k=5)
sparse = sparse_search(hybrid_query, CORPUS + NOISE_DOCS, top_k=5)
fused = rrf_fuse([dense, sparse], top_k=5)

In [17]:
fused

[{'id': 'service',
  'source': 'service.md',
  'text': 'A Service gives stable networking for Pods. ClusterIP is internal, NodePort exposes a port on nodes, and LoadBalancer asks the cloud provider for an external endpoint.',
  'score': 2,
  'rrf_score': 0.032018442622950824},
 {'id': 'pods',
  'source': 'pods.md',
  'text': 'A Kubernetes Pod is the smallest deployable unit. Containers in a Pod share network, storage volumes, and lifecycle. Use kubectl describe pod and kubectl logs to debug Pod behavior.',
  'score': 2,
  'rrf_score': 0.03200204813108039},
 {'id': 'taints',
  'source': 'taints.md',
  'text': 'Taints repel Pods from nodes. Tolerations allow selected Pods to schedule onto tainted nodes. A NoSchedule taint blocks Pods that lack a matching toleration.',
  'score': 5,
  'rrf_score': 0.03177805800756621},
 {'id': 'probes',
  'source': 'probes.md',
  'text': 'Readiness probes decide when a Pod can receive traffic. Liveness probes restart stuck containers. Startup probes prote